In [1]:
import util
import log 
import argparse
import matplotlib.pyplot as plt
import cv2
import numpy as np

In [18]:
roi_logs = util.load_roi_log('./roi_log_7.csv')

frames = []
cap = cv2.VideoCapture('videos/test_cam_2_1450_1900.mp4')
while (cap.isOpened()):
    ret, frame = cap.read()
    if ret:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    else:
        break
cap.release()

In [106]:
import tqdm
import copy
from scipy.spatial import distance


'''
for i, roi_log in enumerate(roi_logs):
    if roi_log.frameIndex == index:
        frame = frames[index]
        location = roi_log.paddedLoc
        util.draw_rect(frame, location, (0,0,0))
        util.write_text(frame, location, str(i))
'''

# copy all frames
draw_frames = []
for frame in frames:
    draw_frames.append(frame.copy())

# extract corners and save at roilogk
corner_index = [0]
index = 1
roi_logs_indexed = [roi_log for roi_log in roi_logs if roi_log.frameIndex == index]
corners = np.empty((0,1,2))
for i,roi_log in enumerate(roi_logs_indexed):
    l,t,r,b = roi_log.origLoc
    gray = cv2.cvtColor(draw_frames[index][t:b,l:r], cv2.COLOR_RGB2GRAY)
    corner = cv2.goodFeaturesToTrack(gray, 100, 0.01, 5, blockSize=3, useHarrisDetector=False, k=0.03)
    corner_index.append(corner_index[-1]+len(corner))
    for i in range(len(corner)):
        corner[i] = np.array([(np.array(corner[i][0])+np.array([l,t]))])
    roi_log.corner = corner
    corners = np.concatenate((corners, corner), axis=0)

'''
for i in corners:
    cv2.circle(draw_frames[index], tuple(map(int,(i[0]))), 1, (255,0,0), 2)
    #cv2.circle(draw_frames[index], tuple(map(int,(i[0]))), 1, (255,0,0), 2)
#fig, ax = plt.subplots(figsize=(20,20))
#plt.axis('off')
#plt.imshow(draw_frames[index])
'''

gray_frame0 = cv2.cvtColor(frames[index], cv2.COLOR_BGR2GRAY)
gray_frame1 = cv2.cvtColor(frames[index+1], cv2.COLOR_BGR2GRAY)
corners = corners.astype(np.float32)
next_points, status, err = cv2.calcOpticalFlowPyrLK(
    gray_frame0, gray_frame1, corners, None,
    winSize=(15,15), criteria=(cv2.TERM_CRITERIA_COUNT + cv2.TERM_CRITERIA_EPS, 10, 0.03), maxLevel=2
)

count = 0

person_id = 6
person_index = slice(corner_index[person_id],corner_index[person_id+1])
person_index = slice(corner_index[0],corner_index[-1])
dists = []

for i,j in zip(corners[person_index], next_points[person_index]):
    frame = draw_frames[index]
    prev_point = tuple(map(int,(i[0])))
    next_point = tuple(map(int,(j[0])))
    
    #dist = distance.euclidean(prev_point, next_point)
    dist = (np.array(next_point) - np.array(prev_point))
    dists.append(dist)


dists = np.array(dists)
xs, ys = dists[:,0], dists[:,1]
print(max(xs), max(ys))
bins = np.arange(0,max(xs),0.03)
hist, bins = np.histogram(xs,bins)
#plt.hist(hist)
#print(hist)

bins = np.arange(0,max(ys),0.03)
hist, bins = np.histogram(ys,bins)
#print(hist)

'''
print(hist)
q1 = np.quantile(dists, 0.25)
q2 = np.quantile(dists, 0.5)
q3 = np.quantile(dists, 0.75)
'''


for i,j in zip(corners[person_index], next_points[person_index]):
    frame = draw_frames[index]
    prev_point = tuple(map(int,(i[0])))
    next_point = tuple(map(int,(j[0])))
    #dist = distance.euclidean(prev_point, next_point)
    dist = (np.array(next_point) - np.array(prev_point))
    cv2.arrowedLine(frame, tuple(map(int,(i[0]))), tuple(map(int,(j[0]))), (0,255,0))
    cv2.circle(frame, prev_point,  0, (255,0,0), 2)
    cv2.circle(frame, next_point, 0, (0,0,255), 2)
    #frame[prev_point[1],prev_point[0]] = [255,0,0]
    #frame[next_point[1],next_point[0]] = [0,0,255]



# extract roi from frame and draw feature points
rois = []
for roi_log in roi_logs_indexed:
    l,t,r,b = roi_log.paddedLoc
    frame = draw_frames[index].copy()
    #cv2.arrowedLine(frame, ((l+r)/2,(t+b)/2), , tuple(map(int,(j[0]))), (0,255,0))
    for i in roi_log.corner:
        pass
        #cv2.circle(frame, tuple(map(int,(i[0]))), 1, (255,0,0), 2)
    roi = frame[t:b,l:r]
    rois.append(roi)


rows = 3
cols = 4
fig, ax = plt.subplots(nrows=rows, ncols=cols, figsize=(30,30))

for i, roi in enumerate(rois[:rows*cols]):
    ax.ravel()[i].imshow(roi)
    ax.ravel()[i].set_title(i)
    ax.ravel()[i].set_axis_off()
plt.tight_layout()
plt.show()


2 2
